# Idealista Barcelona — Detail Scraper (Zyte HTTP)

Sends each URL to the Zyte API HTTP tier (~$1.27/1k), decodes the response body, and parses with BeautifulSoup. Runs 5 workers in parallel. Resumes from existing output CSV.

**Setup:** set your API key in the config cell, or export `ZYTE_API_KEY` as an environment variable before starting Jupyter.

In [ ]:
# %pip install requests beautifulsoup4 pandas tqdm lxml

In [ ]:
import base64
import csv
import hashlib
import json
import os
import re
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

In [ ]:
# ── config ────────────────────────────────────────────────────────────────────
ZYTE_API_KEY   = os.environ.get("ZYTE_API_KEY", "")   # or paste key here as a string
ZYTE_ENDPOINT  = "https://api.zyte.com/v1/extract"

DATA_DIR       = Path("data")
URLS_CSV       = DATA_DIR / "idealista_barcelona_sale_urls.csv"
OUTPUT_CSV     = DATA_DIR / "idealista_barcelona_sale_properties_details.csv"
CACHE_DIR      = DATA_DIR / "html_cache" / "detail_pages"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

WORKERS        = 5       # concurrent Zyte requests; safe to raise to 10
MAX_DETAILS    = None    # set to e.g. 20 for a test run; None = all
USE_CACHE      = True    # reuse cached HTML from previous browser runs
SAVE_EVERY     = 25      # checkpoint to CSV every N completed rows
MAX_RETRIES    = 3       # retry failed Zyte requests this many times
RETRY_DELAY    = 5       # seconds between retries

COLUMNS = [
    "propertyCode", "Link", "district", "neighborhood",
    "price", "size", "bed", "br", "floor",
    "address", "latitude", "longitude", "x", "y",
    "url", "description", "scraped_at",
]

assert ZYTE_API_KEY, "Set ZYTE_API_KEY above or export it as an environment variable"

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

def now_utc():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def clean(v):
    if v is None:
        return None
    v = re.sub(r"\s+", " ", str(v)).strip()
    return v or None

def as_number(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return v
    m = re.search(r"-?\d+(?:[.,]\d+)?", str(v).replace(".", ""))
    if not m:
        return None
    n = float(m.group(0).replace(",", "."))
    return int(n) if n.is_integer() else n

def property_code(url):
    m = re.search(r"/inmueble/(\d+)/", str(url))
    return m.group(1) if m else None

def cache_path(url):
    digest = hashlib.sha1(url.encode()).hexdigest()[:16]
    code = property_code(url)
    stem = f"{code}_{digest}" if code else digest
    return CACHE_DIR / f"{stem}.html"

def load_cache(url):
    p = cache_path(url)
    if USE_CACHE and p.exists():
        return p.read_text(encoding="utf-8")
    return None

def save_cache(url, html):
    if USE_CACHE and html:
        cache_path(url).write_text(html, encoding="utf-8")

In [ ]:
# ── zyte fetch ────────────────────────────────────────────────────────────────

SESSION = requests.Session()
SESSION.auth = (ZYTE_API_KEY, "")

def zyte_fetch(url):
    """Fetch a URL via Zyte HTTP tier. Returns decoded HTML string or None."""
    cached = load_cache(url)
    if cached is not None:
        return cached

    payload = {"url": url, "httpResponseBody": True}
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = SESSION.post(ZYTE_ENDPOINT, json=payload, timeout=60)
            if resp.status_code == 200:
                body_b64 = resp.json().get("httpResponseBody", "")
                html = base64.b64decode(body_b64).decode("utf-8", errors="replace")
                save_cache(url, html)
                return html
            elif resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", RETRY_DELAY * attempt))
                log(f"Rate limited on {url}, waiting {wait}s (attempt {attempt})")
                time.sleep(wait)
            else:
                log(f"HTTP {resp.status_code} on {url} (attempt {attempt})")
                time.sleep(RETRY_DELAY * attempt)
        except Exception as exc:
            log(f"Error fetching {url}: {exc} (attempt {attempt})")
            time.sleep(RETRY_DELAY * attempt)
    return None

In [ ]:
# ── parsing ───────────────────────────────────────────────────────────────────

def json_walk(obj):
    if isinstance(obj, dict):
        yield obj
        for v in obj.values():
            yield from json_walk(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from json_walk(v)

def load_jsonld(soup):
    out = []
    for tag in soup.select("script[type='application/ld+json']"):
        raw = tag.string or tag.get_text()
        if not raw:
            continue
        try:
            out.extend(json_walk(json.loads(raw)))
        except Exception:
            pass
    return out

def schema_value(objects, *keys):
    keys_lower = {k.lower() for k in keys}
    for obj in objects:
        for k, v in obj.items():
            if k.lower() in keys_lower and v not in (None, "", []):
                return v
    return None

def schema_address(objects):
    v = schema_value(objects, "address")
    if isinstance(v, dict):
        parts = [v.get(k) for k in ["streetAddress", "addressLocality", "addressRegion", "postalCode"]]
        return clean(", ".join(str(x) for x in parts if x))
    return clean(v)

def schema_geo(objects):
    for obj in objects:
        if "latitude" in obj and "longitude" in obj:
            return obj["latitude"], obj["longitude"]
        geo = obj.get("geo")
        if isinstance(geo, dict) and "latitude" in geo:
            return geo["latitude"], geo["longitude"]
    return None, None

def regex_geo(html):
    # JSON-style
    for i, pat in enumerate([
        r'"latitude"\s*:\s*([\-\d.]+)\s*,\s*"longitude"\s*:\s*([\-\d.]+)',
        r'"longitude"\s*:\s*([\-\d.]+)\s*,\s*"latitude"\s*:\s*([\-\d.]+)',
    ]):
        m = re.search(pat, html, re.I | re.S)
        if m:
            return (m.group(2), m.group(1)) if i == 1 else (m.group(1), m.group(2))
    # Google Static Maps URL: center=41.3922%2C2.1754
    m = re.search(r'center=([\-\d.]+)(?:%2C|,)([\-\d.]+)', html, re.I)
    if m:
        return m.group(1), m.group(2)
    return None, None

def visible(soup, *selectors):
    for sel in selectors:
        node = soup.select_one(sel)
        if node:
            t = clean(node.get_text(" ", strip=True))
            if t:
                return t
    return None

def parse_features(text):
    text = clean(text) or ""
    out = {"size": None, "bed": None, "br": None, "floor": None}
    m = re.search(r"([\d.,]+)\s*m[²2]", text, re.I)
    if m:
        out["size"] = as_number(m.group(1))
    m = re.search(r"(\d+)\s*bed", text, re.I)
    if m:
        out["bed"] = int(m.group(1))
    m = re.search(r"(\d+)\s*bath", text, re.I)
    if m:
        out["br"] = int(m.group(1))
    for pat in [r"(\d+)(?:st|nd|rd|th)?\s*floor", r"floor\s*(\d+)",
                r"(ground floor|basement|semi-basement|mezzanine|top floor)"]:
        m = re.search(pat, text, re.I)
        if m:
            out["floor"] = m.group(1).lower()
            break
    return out

def parse_district_neighborhood(address):
    # Idealista format: '[Type] in [Street,] [Barrio], Barcelona'
    # district is never in the address; neighborhood = the barrio part
    if not address:
        return None, None
    text = re.sub(r'^[^,]+?\bin\b\s*', '', str(address), flags=re.I).strip()
    parts = [clean(x) for x in text.split(",")]
    parts = [x for x in parts if x and x.lower() not in ("barcelona", "barcelona-barcelona")]
    if len(parts) >= 2:
        return None, parts[-1]   # last non-Barcelona part is the barrio
    if len(parts) == 1:
        return None, parts[0]
    return None, None

def detail_features_text(soup, fallback=None):
    chunks = [x.get_text(" ", strip=True) for x in soup.select(
        ".details-property_features li, .info-features span, "
        ".details-property-feature-one, .details-property-feature-two, .item-detail"
    )]
    return clean(" | ".join(chunks)) or fallback

def parse_detail(html, url, search_row):
    soup = BeautifulSoup(html, "lxml")
    objects = load_jsonld(soup)
    features = parse_features(detail_features_text(soup, search_row.get("details_search")))

    lat, lon = schema_geo(objects)
    if not lat:
        lat, lon = regex_geo(html)

    address = (
        schema_address(objects)
        or visible(soup, "span.main-info__title-main", ".main-info__title-main", "h1")
        or search_row.get("address_search")
    )
    district, neighborhood = parse_district_neighborhood(
        address or search_row.get("address_search")
    )

    price = (
        clean(schema_value(objects, "price"))
        or visible(soup, "span.info-data-price", ".info-data-price", "span.item-price")
        or search_row.get("price_search")
    )

    description = (
        clean(schema_value(objects, "description"))
        or visible(soup, "div.comment", ".adCommentsLanguage", "#details .comment",
                   "[class*='description']")
        or search_row.get("description_search")
    )

    size_schema = schema_value(objects, "floorSize", "size", "area")
    if isinstance(size_schema, dict):
        size_schema = size_schema.get("value") or size_schema.get("amount")

    return {
        "propertyCode": search_row.get("propertyCode") or property_code(url),
        "Link": "LINK",
        "district": district,
        "neighborhood": neighborhood,
        "price": price,
        "size": as_number(size_schema) or features["size"],
        "bed": as_number(schema_value(objects, "numberOfBedrooms", "numberOfRooms")) or features["bed"],
        "br": as_number(schema_value(objects, "numberOfBathroomsTotal", "numberOfBathrooms")) or features["br"],
        "floor": clean(schema_value(objects, "floorLevel", "floor")) or features["floor"],
        "address": address,
        "latitude": lat,
        "longitude": lon,
        "x": lon,
        "y": lat,
        "url": url,
        "description": description,
        "scraped_at": now_utc(),
    }

In [ ]:
# ── main loop ─────────────────────────────────────────────────────────────────

_lock = threading.Lock()
_rows = []
_save_counter = 0

def checkpoint():
    pd.DataFrame(_rows).reindex(columns=COLUMNS).drop_duplicates("url", keep="last").to_csv(
        OUTPUT_CSV, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL
    )

def process(search_row):
    global _save_counter
    url = search_row["url"]
    html = zyte_fetch(url)
    if not html:
        return None
    row = parse_detail(html, url, search_row)
    with _lock:
        _rows.append(row)
        _save_counter += 1
        if _save_counter % SAVE_EVERY == 0:
            checkpoint()
    return row


def run():
    global _rows, _save_counter

    urls_df = pd.read_csv(URLS_CSV).dropna(subset=["url"]).drop_duplicates("url")
    log(f"URLs loaded: {len(urls_df):,}")

    done = set()
    if OUTPUT_CSV.exists():
        existing = pd.read_csv(OUTPUT_CSV)
        _rows = existing.reindex(columns=COLUMNS).to_dict("records")
        done = set(existing["url"].dropna().astype(str))
        log(f"Resuming: {len(done):,} already done")
    else:
        _rows = []
        log("Starting fresh")

    todo = urls_df[~urls_df["url"].astype(str).isin(done)].copy()
    if MAX_DETAILS:
        todo = todo.head(MAX_DETAILS)
    log(f"To scrape: {len(todo):,}")

    todo_records = todo.to_dict("records")
    _save_counter = 0
    skipped = 0

    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = {pool.submit(process, row): row["url"] for row in todo_records}
        with tqdm(total=len(futures), desc="Zyte fetch") as pbar:
            for future in as_completed(futures):
                result = future.result()
                if result is None:
                    skipped += 1
                pbar.update(1)

    checkpoint()  # final save
    final = pd.DataFrame(_rows).reindex(columns=COLUMNS).drop_duplicates("url", keep="last")
    log(f"Done: {len(final):,} rows saved to {OUTPUT_CSV} | skipped: {skipped}")
    return final


properties_df = run()
properties_df.head()

In [ ]:
# ── quality check ─────────────────────────────────────────────────────────────
df = pd.read_csv(OUTPUT_CSV)
display(df.head())
display(df.isna().mean().sort_values(ascending=False).to_frame("missing_share"))
print(f"Rows: {len(df):,}")
print(f"Unique property codes: {df['propertyCode'].nunique():,}")
print(f"Rows with lat/lon: {df[['latitude','longitude']].notna().all(axis=1).sum():,}")